# Field Shift · Punto 4: años que rindieron distinto de lo esperado

**Qué ve el productor:** en la historia de su condado, los años que se salieron de lo normal, con el motivo más probable. Por ejemplo:

> "In 2012, corn in this county yielded about {X} % less than in a normal year. July heat tends to explain most of it."

**Cómo se hace**, con el modelo del punto 2 (no se entrena nada nuevo):

1. **Rinde de un año normal:** el modelo predice el rinde de cada año usando el **clima típico** del condado en vez del clima de ese año. Así queda lo que se esperaba con ese suelo, esa rotación y esa tecnología.
2. **Diferencia:** rinde real − rinde de un año normal. Si se aparta más que `UMBRAL_PCT`, el año se marca.
3. **Por qué:** cuánto aportó cada grupo de clima respecto de un año normal (calor de julio, primavera húmeda, etc.), con los valores SHAP del modelo. El grupo que más pesó es la causa principal.

**Antes de correrla:** correr `punto2_rotaciones.ipynb` con maíz y con soja (crea los modelos en `backend/ML/modelos/` y `backend/Datos/tabla_modelado.csv`). Si faltan, cada paso dice "en espera".

**Cómo usarla:** elegir `CULTIVO` en el paso 1, correr todo y repetir con el otro cultivo. Las salidas de los dos se juntan en el mismo archivo.

## 0. Librerías

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

## 1. Parámetros

- `UMBRAL_PCT`: cuánto tiene que apartarse el rinde real del de un año normal para marcar el año. Con 10 %, en maíz es unos 18 bu/ac, parecido al error promedio del modelo del punto 2: menos que eso no se distingue del ruido.
- `GRUPOS_CLIMA`: los mismos grupos y nombres que propone el punto 3 (`PUNTO2.md`, sección "Para el punto 3"). Si cambian allá, cambialos acá.
- Solo se usan los años de desarrollo (2009–2023). 2024–2025 son la prueba final del punto 2 y no se miran todavía.

In [ ]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
SALIDA = RAIZ / "backend" / "Datos"

CULTIVO = "maiz"            # "maiz" o "soja": se corre una vez por cultivo
UMBRAL_PCT = 10             # % de diferencia para marcar un año (solo con MARCAR = "pct")
# Cómo se marca un año: "banda" = el rinde real cae fuera de la banda p10–p90 del modelo alrededor de un año normal
# (la misma banda del punto 2, por estado); "pct" = se aparta más que UMBRAL_PCT. Con "pct" se marcaba 1 de cada 3 años.
MARCAR = "banda"
# "No explicado" con las predicciones fuera de muestra del punto 2 (cada año, con un modelo que no lo vio).
# Con el modelo final, lo no explicado queda achicado porque el modelo ya vio ese año.
USAR_OOF = True
# Causa: un grupo de clima es "la causa" si el clima explica al menos esta fracción de la diferencia con un año normal,
# en la misma dirección. Con 0,5 la frase del front ("tends to explain most of it") es literal.
FRACCION_CAUSA = 0.5
ANIOS_PRUEBA = [2024, 2025]
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa", "20": "Kansas", "27": "Minnesota",
                  "29": "Missouri", "31": "Nebraska", "38": "North Dakota", "39": "Ohio", "46": "South Dakota", "55": "Wisconsin"}

# Solo grupos de clima: suelo y rotación no cambian de un año a otro dentro del condado
GRUPOS_CLIMA = {
    "calor_julio": ["tmax_jul_c", "dias_tmax_gt30_jul", "dias_tmax_gt35_07"],
    "agua_julio": ["prec_jul_mm", "deficit_mm_07"],
    "calor_agosto": ["tmax_c_08"],
    "agua_agosto": ["prec_ago_mm", "deficit_mm_08"],
    "temporada": ["prec_temporada_mm", "gdd_temporada", "deficit_temporada_mm"],
    "primavera_humeda": ["prec_power_mm_04", "prec_power_mm_05", "dias_lluvia_gt50_04", "dias_lluvia_gt50_05",
                         "gwettop_04", "gwettop_05", "gwetroot_04", "gwetroot_05"],
}

ruta_modelo = RAIZ / "backend" / "ML" / "modelos" / f"punto2_rotaciones_{CULTIVO}.joblib"
ruta_tabla = SALIDA / "tabla_modelado.csv"
HAY_MODELO = ruta_modelo.exists() and ruta_tabla.exists()
print("Cultivo:", CULTIVO, "| ¿Está el modelo del punto 2?", HAY_MODELO)
if not HAY_MODELO:
    print("Falta correr punto2_rotaciones.ipynb con este cultivo. La notebook queda en espera.")

## 2. Cargar el modelo y la tabla del punto 2

El modelo predice la **anomalía** (rinde − tendencia del estado). Para volver a bu/ac se le suma la tendencia, que viene guardada junto con el modelo.

In [ ]:
if HAY_MODELO:
    paquete = joblib.load(ruta_modelo)
    tabla = pd.read_csv(ruta_tabla, dtype={"fips": str, "estado": str})
    tabla = tabla[(tabla["cultivo"] == CULTIVO) & ~tabla["anio"].isin(ANIOS_PRUEBA)].reset_index(drop=True)
    COLUMNAS_CLIMA = [c for cols in GRUPOS_CLIMA.values() for c in cols if c in paquete["features"]]
    print(f"Modelo: {type(paquete['modelo'].named_steps['modelo']).__name__} | datos {paquete['version_datos']} "
          f"| {len(tabla)} filas condado-año | {len(COLUMNAS_CLIMA)} columnas de clima")
else:
    print("En espera del modelo del punto 2.")

## 3. Funciones del punto 2

Copias de `separar_rotacion` y de la tendencia del punto 2, para usar el modelo guardado igual que allá. Si cambian en el punto 2, cambialas acá.

In [ ]:
def separar_rotacion(datos, medias):
    d = datos.copy()
    for c in paquete["rotacion"]:
        media = d["fips"].map(medias[c]).fillna(d.groupby("fips")[c].transform("mean"))
        d[f"{c}_media"] = media
        d[f"{c}_desvio"] = d[c] - media
    return d


def aplicar_tendencia(coefs, datos):
    """Copia del punto 2: la recta del estado + el nivel propio del condado, si el modelo lo trae."""
    valor = np.array([np.polyval(coefs["por_estado"].get(e, coefs["global"]), a)
                      for e, a in zip(datos["estado"], datos["anio"])])
    if "nivel_condado" in coefs:
        valor = valor + datos["fips"].map(coefs["nivel_condado"]).fillna(0).to_numpy()
    return valor


def predecir_con_contribuciones(datos):
    """Rinde predicho (bu/ac) y la contribución SHAP de cada columna (bu/ac)."""
    x = separar_rotacion(datos, paquete["medias"])[paquete["features"]]
    prep = paquete["modelo"].named_steps["prep"]
    arboles = paquete["modelo"].named_steps["modelo"]
    contrib = arboles.predict(prep.transform(x), pred_contrib=True)
    nombres = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
    shap = pd.DataFrame(contrib[:, :-1], columns=nombres, index=datos.index)
    return contrib.sum(axis=1) + aplicar_tendencia(paquete["tendencia"], datos), shap

## 4. Rinde de un año normal

Para cada condado y año se arma una copia con el **clima típico del condado** (la mediana de sus años 2009–2023) y todo lo demás igual: el mismo suelo, la misma rotación y la misma tendencia de ese año. Lo que predice el modelo con esa copia es el rinde de un año normal.

Se usa la mediana y no el promedio para que un año extremo como 2012 no corra lo que se considera normal.

In [ ]:
if HAY_MODELO:
    normal = tabla.copy()
    clima_tipico = tabla.groupby("fips")[COLUMNAS_CLIMA].median()
    normal[COLUMNAS_CLIMA] = clima_tipico.loc[tabla["fips"]].to_numpy()

    pred_real, shap_real = predecir_con_contribuciones(tabla)
    pred_normal, shap_normal = predecir_con_contribuciones(normal)

    res = tabla[["fips", "anio", "estado", "cultivo", "rend_bu_ac"]].copy()
    res["rend_anio_normal_bu_ac"] = pred_normal
    res["rend_predicho_bu_ac"] = pred_real
    res["diferencia_bu_ac"] = res["rend_bu_ac"] - res["rend_anio_normal_bu_ac"]
    res["diferencia_pct"] = res["diferencia_bu_ac"] / res["rend_anio_normal_bu_ac"] * 100
    display(res.describe()[["rend_bu_ac", "rend_anio_normal_bu_ac", "diferencia_pct"]].round(1))
else:
    print("En espera del modelo del punto 2.")

## 5. Por qué: el clima de ese año contra el de un año normal

Para cada grupo de clima se resta su contribución en el año real menos su contribución en el año normal. Eso dice **cuántos bu/ac le sumó o le restó ese grupo** respecto de un año normal.

Se separan dos partes de la diferencia:

- **explicada por el clima:** la suma de los grupos (lo que el modelo atribuye al clima del año);
- **no explicada:** rinde real − rinde predicho con el clima real. Es lo que el modelo no ve: granizo, inundaciones, plagas, errores del dato o simplemente error del modelo.

La **causa principal** es el grupo con más peso, siempre que vaya en la misma dirección que la diferencia. Si lo no explicado pesa más que todo el clima junto, la causa queda como `no_explicado`.

In [ ]:
if HAY_MODELO:
    efecto = pd.DataFrame({g: (shap_real[[c for c in cols if c in shap_real]].sum(axis=1)
                               - shap_normal[[c for c in cols if c in shap_normal]].sum(axis=1))
                           for g, cols in GRUPOS_CLIMA.items()})
    res["explicado_clima_bu_ac"] = efecto.sum(axis=1)
    res["no_explicado_bu_ac"] = res["rend_bu_ac"] - res["rend_predicho_bu_ac"]
    ruta_oof = SALIDA / "predicciones_oof.csv"
    if USAR_OOF and ruta_oof.exists():
        oof = pd.read_csv(ruta_oof, dtype={"fips": str})
        oof = oof[oof["cultivo"] == CULTIVO].set_index(["fips", "anio"])["pred_oof"]
        pred_oof = pd.Series(list(zip(res["fips"], res["anio"]))).map(oof).to_numpy()
        hay = ~np.isnan(pred_oof)
        res.loc[hay, "no_explicado_bu_ac"] = (res["rend_bu_ac"] - pred_oof)[hay]
        print(f"No explicado con predicciones fuera de muestra en {hay.mean() * 100:.0f} % de las filas")

    signo = np.sign(res["diferencia_bu_ac"])
    a_favor = efecto.mul(signo, axis=0)                     # positivo = empuja en la dirección de la diferencia
    res["causa_principal"] = a_favor.idxmax(axis=1)
    columna_causa = efecto.columns.get_indexer(res["causa_principal"])
    res["efecto_causa_bu_ac"] = efecto.to_numpy()[np.arange(len(efecto)), columna_causa]
    clima_explica = ((np.sign(res["explicado_clima_bu_ac"]) == signo)
                     & (res["explicado_clima_bu_ac"].abs() >= FRACCION_CAUSA * res["diferencia_bu_ac"].abs()))
    sin_causa = (a_favor.max(axis=1) <= 0) | ~clima_explica
    res.loc[sin_causa, "causa_principal"] = "no_explicado"
    res.loc[sin_causa, "efecto_causa_bu_ac"] = res.loc[sin_causa, "no_explicado_bu_ac"]

    if MARCAR == "banda":
        # Banda del punto 2 por estado (si el modelo la trae); si no, la de todos
        q_estado = paquete.get("q_por_estado") or {}
        q_bajo = res["estado"].map(lambda e: q_estado.get(e, (paquete["q10"], paquete["q90"]))[0])
        q_alto = res["estado"].map(lambda e: q_estado.get(e, (paquete["q10"], paquete["q90"]))[1])
        res["marcado"] = (res["diferencia_bu_ac"] < q_bajo) | (res["diferencia_bu_ac"] > q_alto)
        criterio = "fuera de la banda p10–p90 del modelo"
    else:
        res["marcado"] = res["diferencia_pct"].abs() >= UMBRAL_PCT
        criterio = f"umbral {UMBRAL_PCT} %"
    res["direccion"] = np.where(res["diferencia_bu_ac"] < 0, "peor", "mejor")
    print(f"Años marcados: {res['marcado'].mean() * 100:.0f} % de los condado-año ({criterio})")
    display(pd.crosstab(res.loc[res["marcado"], "causa_principal"], res.loc[res["marcado"], "direccion"], margins=True))
else:
    print("En espera del modelo del punto 2.")

## 6. Chequeo: ¿reconoce los años que todos conocen?

Si el método funciona, tiene que encontrar los años famosos del Corn Belt:

- **2012:** sequía y calor en julio. La mayoría de los condados tienen que salir marcados como `peor`, con causa de calor o falta de agua.
- **2019:** primavera muy húmeda, siembra atrasada. Tendría que aparecer `primavera_humeda` como causa en bastantes condados.

Por año: qué % de condados quedó marcado, la diferencia mediana y la causa más común entre los marcados.

In [ ]:
if HAY_MODELO:
    marcados = res[res["marcado"]]
    por_anio = pd.DataFrame({
        "% condados marcados": res.groupby("anio")["marcado"].mean() * 100,
        "diferencia mediana %": res.groupby("anio")["diferencia_pct"].median(),
        "causa más común (marcados)": marcados.groupby("anio")["causa_principal"].agg(lambda s: s.mode().iat[0]),
    })
    display(por_anio.round(0))

    r2012 = res[res["anio"] == 2012]
    ok_2012 = (r2012["marcado"] & (r2012["direccion"] == "peor")).mean()
    calor_agua = r2012.loc[r2012["marcado"], "causa_principal"].isin(["calor_julio", "agua_julio", "temporada",
                                                                      "calor_agosto", "agua_agosto"]).mean()
    print(f"2012: {ok_2012 * 100:.0f} % de los condados marcados como peor; "
          f"en {calor_agua * 100:.0f} % de esos la causa es calor o falta de agua.")
    if ok_2012 < 0.5:
        print("ATENCIÓN: no reconoce la sequía de 2012. Revisar antes de mostrarlo.")
else:
    print("En espera del modelo del punto 2.")

## 7. Ejemplo: la historia de un condado

Lo que vería el productor en la línea de tiempo: cada año con su diferencia y, si está marcado, la causa.

In [ ]:
FIPS_EJEMPLO = "19169"   # Story, Iowa

if HAY_MODELO:
    ej = res[res["fips"] == FIPS_EJEMPLO].set_index("anio")
    display(ej[["rend_bu_ac", "rend_anio_normal_bu_ac", "diferencia_pct", "marcado", "causa_principal",
                "efecto_causa_bu_ac"]].round(1))
else:
    print("En espera del modelo del punto 2.")

## 8. Guardar

`backend/Datos/anomalias_anio.csv`, una fila por condado, año y cultivo:

| Columna | Qué es |
|---|---|
| `fips, anio, estado, cultivo` | Claves |
| `rend_bu_ac` | Rinde real (NASS) |
| `rend_anio_normal_bu_ac` | Rinde predicho con el clima típico del condado |
| `rend_predicho_bu_ac` | Rinde predicho con el clima real de ese año |
| `diferencia_bu_ac`, `diferencia_pct` | Real − año normal |
| `marcado` | `True` si `diferencia_pct` se aparta `UMBRAL_PCT` o más |
| `direccion` | `peor` o `mejor` que un año normal |
| `causa_principal` | Id del grupo de clima que más pesó, o `no_explicado` |
| `efecto_causa_bu_ac` | Cuántos bu/ac aportó esa causa respecto de un año normal |
| `explicado_clima_bu_ac` | Suma de todos los grupos de clima |
| `no_explicado_bu_ac` | Real − predicho con el clima real |
| `ef_<grupo>_bu_ac` | Aporte de cada grupo de clima, para quien quiera más detalle |

Los nombres para la app de cada `causa_principal` van en `textos.json` (punto 3).

In [ ]:
def guardar_por_cultivo(df, ruta):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if ruta.exists():
        viejo = pd.read_csv(ruta, dtype={"fips": str, "estado": str})
        df = pd.concat([viejo[viejo["cultivo"] != CULTIVO], df], ignore_index=True)
    df.to_csv(ruta, index=False)


if HAY_MODELO:
    salida = pd.concat([res, efecto.add_prefix("ef_").add_suffix("_bu_ac")], axis=1)
    salida = salida.round({c: 1 for c in salida.columns if salida[c].dtype.kind == "f"})
    guardar_por_cultivo(salida, SALIDA / "anomalias_anio.csv")
    print(f"Guardadas {len(salida)} filas en backend/Datos/anomalias_anio.csv")
else:
    print("En espera del modelo del punto 2.")

## 9. Archivo para el front

Arma `backend/Datos/front/anomalias.json` con el formato acordado con el front: para cada condado, cultivo (`corn`, `soy`) y año, el rinde, el de un año normal (`usual_bu_ac`), si estuvo por encima, parecido o por debajo (`vs_usual`), la causa y un titular corto. El script del front lo pega en `county/<fips>.json`. Los titulares dependen de si el año fue mejor o peor: un `calor_julio` en un año mejor es un julio templado ("Mild July").

In [ ]:
import json
from datetime import datetime

import pandas as pd

# Paso 9: backend/Datos/front/anomalias.json para la línea de tiempo del front (pantalla 2).
# Lee el CSV del paso 8 (maíz y soja juntos), así que conviene correrlo después de las dos corridas.
# Formato acordado con el front: counties[fips][corn|soy][anio] → anomaly. El script del front
# (scripts/build_front_data.py) lo pega en county/<fips>.json, en cada año.

CULTIVO_FRONT = {"maiz": "corn", "soja": "soy"}
NOMBRE_EN = {"maiz": "Corn", "soja": "Soybean"}

# Titular corto (para F19 y las marcas de F07) y frase de la causa, según la dirección del año.
# "peor" = rindió menos que un año normal; "mejor" = más. Siempre "tends to", nunca "because".
FRASES = {
    "calor_julio": {
        "peor": ("Hot July", "July heat (corn flowering) tends to explain most of it."),
        "mejor": ("Mild July", "A mild July (corn flowering) tends to explain most of it."),
    },
    "agua_julio": {
        "peor": ("July water stress", "July rain and water balance tend to explain most of it."),
        "mejor": ("Good July moisture", "Good July rain and water balance tend to explain most of it."),
    },
    "calor_agosto": {
        "peor": ("Hot August", "August heat (soybean pod filling) tends to explain most of it."),
        "mejor": ("Mild August", "A mild August (soybean pod filling) tends to explain most of it."),
    },
    "agua_agosto": {
        "peor": ("August water stress", "August rain and water balance tend to explain most of it."),
        "mejor": ("Good August moisture", "Good August rain and water balance tend to explain most of it."),
    },
    "temporada": {
        "peor": ("Tough season", "Season rain, warmth and water balance (Apr–Sep) tend to explain most of it."),
        "mejor": ("Favorable season", "Season rain, warmth and water balance (Apr–Sep) tend to explain most of it."),
    },
    "primavera_humeda": {
        "peor": ("Wet spring", "A wet spring (it can delay planting) tends to explain most of it."),
        "mejor": ("Favorable spring", "Spring rain and soil moisture tend to explain most of it."),
    },
    "no_explicado": {
        "peor": ("Not explained by weather data",
                 "The weather data doesn't explain most of it (hail, wind, flooding, pests…)."),
        "mejor": ("Not explained by weather data",
                  "The weather data doesn't explain most of it."),
    },
}


def fuente(causa):
    return "USDA NASS" if causa == "no_explicado" else "USDA NASS + NASA POWER"


def anomalia(f):
    """Un año de un cultivo en un condado, en el formato del front."""
    if pd.isna(f.rend_bu_ac) or pd.isna(f.rend_anio_normal_bu_ac):
        return None  # "Missing data"
    marcado = bool(f.marcado)
    vs = ("below" if f.direccion == "peor" else "above") if marcado else "similar"
    if marcado:
        titular, causa_txt = FRASES[f.causa_principal][f.direccion]
        menos_mas = "less" if f.direccion == "peor" else "more"
        texto = (f"{NOMBRE_EN[f.cultivo]} yielded about {abs(round(f.diferencia_pct))}% {menos_mas} "
                 f"than in a normal year here. {causa_txt}")
    else:
        titular = None
        texto = f"{NOMBRE_EN[f.cultivo]} yield was close to a normal year here (within {UMBRAL_PCT}%)."
    return {
        "bu_ac": round(float(f.rend_bu_ac), 1),
        "usual_bu_ac": round(float(f.rend_anio_normal_bu_ac), 1),
        "diff_bu_ac": round(float(f.diferencia_bu_ac), 1),
        "diff_pct": round(float(f.diferencia_pct), 1),
        "vs_usual": vs,
        "flagged": marcado,
        "cause": f.causa_principal if marcado else None,
        "cause_bu_ac": round(float(f.efecto_causa_bu_ac), 1) if marcado else None,
        "headline": titular,
        "text": texto,
        "basis": "model",
        "source": fuente(f.causa_principal) if marcado else "USDA NASS",
    }


ruta_csv = SALIDA / "anomalias_anio.csv"
if ruta_csv.exists():
    anom = pd.read_csv(ruta_csv, dtype={"fips": str, "estado": str})
    faltan = set(anom["causa_principal"]) - set(FRASES)
    assert not faltan, f"Causas sin frase: {faltan}"
    condados = {}
    for f in anom.sort_values(["fips", "cultivo", "anio"]).itertuples():
        a = anomalia(f)
        if a is not None:
            condados.setdefault(f.fips, {}).setdefault(CULTIVO_FRONT[f.cultivo], {})[str(int(f.anio))] = a
    salida_json = {
        "version_contrato": "0.2",
        "generado": datetime.now().isoformat(timespec="seconds"),
        "es_prueba": False,
        "prototype": False,
        "metric": "yield_bu_ac",
        "years": [int(anom["anio"].min()), int(anom["anio"].max())],
        "threshold_pct": UMBRAL_PCT,
        "note": ("'usual_bu_ac' is the model's yield for this county with its typical weather (median "
                 f"{int(anom['anio'].min())}–{int(anom['anio'].max())}). Causes are associations, not proof."),
        "cause_labels": {c: {"worse": v["peor"][0], "better": v["mejor"][0]} for c, v in FRASES.items()},
        "counties": condados,
    }
    (SALIDA / "front").mkdir(parents=True, exist_ok=True)
    (SALIDA / "front" / "anomalias.json").write_text(
        json.dumps(salida_json, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    n = sum(len(v) for c in condados.values() for v in c.values())
    print(f"Guardado backend/Datos/front/anomalias.json: {len(condados)} condados, {n} condado-cultivo-año")
    print(json.dumps(condados.get("19169", {}).get("corn", {}).get("2012"), ensure_ascii=False, indent=1))
else:
    print("En espera: falta backend/Datos/anomalias_anio.csv (correr el paso 8).")

## Próximos pasos

- [x] Archivo para la línea de tiempo del front: `backend/Datos/front/anomalias.json` (paso 9). Falta que el script del front lo lea.
- [x] Frases para cada `causa_principal`, según el año sea mejor o peor (`FRASES`, paso 9).
- [ ] Hacer las predicciones fuera del fold (reentrenando sin cada año), para que el año explicado no esté en el entrenamiento.
- [ ] Cuando se mire la prueba del punto 2, sumar 2024 y 2025.
- [ ] Clasificar cada año como seco, normal o húmedo con la misma regla que el punto 1 (`clase_anio`), cuando esté definida.